In [ ]:

from fastapi import (
    APIRouter,
    HTTPException,
    status,
    Depends,
    Query,
)

from app.api.auth import get_current_user


router = APIRouter(
    prefix="/admin",
    tags=["Admin"],
)


def get_database():
    from app.main import app

    database = getattr(app.state, "database", None)

    if database is None:
        raise RuntimeError("Database is not initialized.")

    return database


def require_admin(
    current_user=Depends(get_current_user),
):
    """
    Require an authenticated administrator.

    Normal student accounts must never access admin endpoints.
    """

    if getattr(current_user, "role", "student") != "admin":
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="Administrator access required.",
        )

    return current_user


def clean(document):
    result = dict(document)
    result.pop("_id", None)
    result.pop("password_hash", None)
    return result


@router.get("/overview")
async def admin_overview(
    current_user=Depends(require_admin),
):
    """Return platform-level administrative analytics."""

    database = get_database()

    return {
        "users": database.collection(
            "users"
        ).count_documents({}),

        "spaces": database.collection(
            "spaces"
        ).count_documents({}),

        "projects": database.collection(
            "projects"
        ).count_documents({}),

        "materials": database.collection(
            "materials"
        ).count_documents({}),

        "conversations": database.collection(
            "conversations"
        ).count_documents({}),

        "assessments": database.collection(
            "assessments"
        ).count_documents({}),

        "activities": database.collection(
            "activities"
        ).count_documents({}),

        "ai_usage_records": database.collection(
            "ai_usage"
        ).count_documents({}),
    }


@router.get("/users")
async def admin_users(
    skip: int = Query(
        default=0,
        ge=0,
    ),
    limit: int = Query(
        default=50,
        ge=1,
        le=200,
    ),
    current_user=Depends(require_admin),
):
    """Return paginated user information."""

    database = get_database()

    users = database.collection("users")

    total = users.count_documents({})

    documents = users.find(
        {},
        {
            "_id": 0,
            "password_hash": 0,
        },
    ).sort(
        "created_at",
        -1,
    ).skip(skip).limit(limit)

    return {
        "total": total,
        "skip": skip,
        "limit": limit,
        "users": [
            clean(document)
            for document in documents
        ],
    }


@router.get("/ai-usage")
async def admin_ai_usage(
    current_user=Depends(require_admin),
):
    """Return AI usage and observability metrics."""

    database = get_database()

    ai_usage = database.collection(
        "ai_usage"
    )

    total_requests = ai_usage.count_documents({})

    successful_requests = ai_usage.count_documents(
        {
            "success": True,
        }
    )

    failed_requests = ai_usage.count_documents(
        {
            "success": False,
        }
    )

    pipeline = [
        {
            "$group": {
                "_id": None,
                "total_tokens": {
                    "$sum": {
                        "$ifNull": [
                            "$total_tokens",
                            0,
                        ]
                    }
                },
                "estimated_cost": {
                    "$sum": {
                        "$ifNull": [
                            "$estimated_cost",
                            0,
                        ]
                    }
                },
                "avg_latency_ms": {
                    "$avg": "$latency_ms",
                },
            }
        }
    ]

    aggregate = list(
        ai_usage.aggregate(pipeline)
    )

    summary = aggregate[0] if aggregate else {}

    return {
        "total_requests": total_requests,
        "successful_requests": successful_requests,
        "failed_requests": failed_requests,
        "total_tokens": summary.get(
            "total_tokens",
            0,
        ),
        "estimated_cost": summary.get(
            "estimated_cost",
            0.0,
        ),
        "average_latency_ms": summary.get(
            "avg_latency_ms",
            0.0,
        ),
    }


@router.get("/system")
async def admin_system(
    current_user=Depends(require_admin),
):
    """Return basic system health information."""

    database = get_database()

    database.client.admin.command("ping")

    return {
        "status": "healthy",
        "database": "mongodb_atlas",
        "database_connection": "healthy",
    }
